### ЗАДАЧА: Пакетная загрузка отгрузок (try/except + custom exceptions)

Из внешней логистической системы приходят строки с отгрузками.
Нужно безопасно распарсить данные, отделить валидные записи от ошибок
и посчитать несколько итоговых метрик.

НЕОБХОДИМО РЕАЛИЗОВАТЬ:

1. Иерархию кастомных исключений:
   - `ShipmentError`
   - `RowFormatError`
   - `WeightError`
   - `PriorityError`
   - `RegionError`.

2. Функцию `parse_shipment(row)`:
   - формат строки: `shipment_id,client,weight,priority,region`
   - `weight` должен быть числом и `> 0`
   - допустимые приоритеты: `standard`, `express`, `vip`
   - допустимые регионы: `RU`, `KZ`, `BY`
   - при ошибке конвертации веса использовать `raise ... from ...`.

3. Функцию `load_shipments(rows)`:
   - вернуть `(shipments, errors)`
   - ошибки хранить как `(row, error_type, message)`
   - не останавливать цикл на первой ошибке.

4. Вывести:
   - число валидных отгрузок,
   - ошибки по типам,
   - суммарный вес только для `express` и `vip`,
   - клиента-лидера по суммарному весу среди валидных записей.

In [1]:
rows = [
    'S-100,Acme,12.5,express,RU',
    'S-101,Beta,0,standard,RU',
    'S-102,Acme,abc,vip,KZ',
    'S-103,Delta,8.5,urgent,BY',
    'S-104,Gamma,15,vip,UZ',
    'S-105,Acme,4.0,standard,KZ',
    'S-106,Beta,9.5,express,BY',
]


class ShipmentError(Exception):
    pass


class RowFormatError(ShipmentError):
    pass


class WeightError(ShipmentError):
    pass


class PriorityError(ShipmentError):
    pass


class RegionError(ShipmentError):
    pass


def parse_shipment(row):
    # TODO: распарсить строку и провалидировать weight, priority, region
    # TODO: при ошибке конвертации weight использовать raise ... from ...
    parts = row.split(",")
    if len(parts) != 5:
        raise RowFormatError("Неверный формат строки")
    shipment_id, client, weight_str, priority, region = parts
    
    try:
        weight = float(weight_str)
        if weight <= 0:
            raise WeightError("Вес должен быть больше 0")
    except ValueError:
        raise WeightError("Вес должен быть числом") from None
    
    if priority not in ("standard", "express", "vip"):
        raise PriorityError("Недопустимый приоритет")
    
    if region not in ("RU", "KZ", "BY"):
        raise RegionError("Недопустимый регион")
    
    return {
        "shipment_id": shipment_id,
        "client": client,
        "weight": weight,
        "priority": priority,
        "region": region
    }

def load_shipments(rows):
    # TODO: вернуть (shipments, errors)
    shipments = []
    errors = []
    
    for row in rows:
        try:
            shipment = parse_shipment(row)
            shipments.append(shipment)
        except ShipmentError as e:
            if isinstance(e, RowFormatError):
                err_type = "RowFormatError"
            elif isinstance(e, WeightError):
                err_type = "WeightError"
            elif isinstance(e, PriorityError):
                err_type = "PriorityError"
            elif isinstance(e, RegionError):
                err_type = "RegionError"
            else:
                err_type = "UnknownError"
            errors.append((row, err_type, str(e)))
            
    return shipments, errors
            


# TODO: вызвать load_shipments(rows)
shipments, errors = load_shipments(rows)
# TODO: вывести число валидных отгрузок и число ошибок
print("Общее число отгрузок:", len(shipments))
print("Общее число ошибок:", len(errors))
print()
# TODO: вывести ошибки по типам
from collections import Counter
error_types = Counter(e[1] for e in errors)
print("Ошибки по типам:")
for etype, count in error_types.items():
    print(f"{etype}: {count}")
print()
# TODO: посчитать premium_weight только для express/vip
premium_weight = sum(s["weight"] for s in shipments if s["priority"] in ("express", "vip"))
print(f"Общий вес для express/vip: {premium_weight}")
# TODO: найти клиента-лидера по суммарному весу
client_weights = {}
for s in shipments:
    client_weights[s["client"]] = client_weights.get(s["client"], 0) + s["weight"]
leader = max(client_weights.items(), key=lambda x: x[1])
print(f"Клиент-лидер по весу: {leader[0]} с весом {leader[1]}")

Общее число отгрузок: 3
Общее число ошибок: 4

Ошибки по типам:
WeightError: 2
PriorityError: 1
RegionError: 1

Общий вес для express/vip: 22.0
Клиент-лидер по весу: Acme с весом 16.5
